# The Pareidolia Paradox — Lunar Surface Classification**Challenge:** Classify 256×256 grayscale lunar images into Depth (0) or Rise (1)  **Key Physics:** Sun azimuth angle causes topographic inversion — we normalize it  **Metric:** Balanced Accuracy  **Pipeline:** CPU preprocessing → GPU training (5-fold EfficientNet-B0) → threshold-optimized submission### Execution Guide| Stage | GPU? | Cells | What ||-------|------|-------|------|| **1** | CPU ⚡ | 0–3 | Setup + azimuth rotation (saves GPU quota) || — | — | — | *Enable GPU, restart kernel, re-run cells 0+1* || **2** | GPU 🔥 | 4–6 | Dataset + Model + Train 5 folds || **3** | GPU | 7–8 | Threshold sweep + Inference → submission.csv |

In [ ]:
# ============================================================
# CELL 0: Environment Setup
# ============================================================
!pip install -q timm albumentations

import torch, torchvision, timm, albumentations, cv2, sklearn
import pandas as pd, numpy as np
from pathlib import Path

cv2.setNumThreads(0)
print(f"PyTorch:        {torch.__version__}")
print(f"Timm:           {timm.__version__}")
print(f"Albumentations: {albumentations.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 1 — Configuration**This is the only cell you need to edit.** Update `dataset_slug` to match your Kaggle dataset.

In [ ]:
# ============================================================
# CELL 1: Configuration — EDIT THIS FOR YOUR SETUP
# ============================================================
from dataclasses import dataclass, field
from typing import List, Tuple

@dataclass
class CFG:
    # ── Kaggle Paths ───────────────────────────────────────
    # ⬇️ CHANGE THIS to your Kaggle dataset slug
    dataset_slug: str = "datasets/vikasgandla1/pareidolia-data"

    @property
    def input_dir(self):
        return Path(f"/kaggle/input/{self.dataset_slug}")
    
    # Dataset subfolders (matches your local structure)
    train_meta_path:  str = "Train/train_metadata.csv"
    test_meta_path:   str = "Test/test_metadata.csv"
    train_img_folder: str = "Train/train_images"
    eval_img_folder:  str = "Test/eval_images"

    # Output dirs (persist across GPU restart)
    work_dir:       Path = Path("/kaggle/working")
    norm_train_dir: Path = Path("/kaggle/working/norm_train")
    norm_eval_dir:  Path = Path("/kaggle/working/norm_eval")
    ckpt_dir:       Path = Path("/kaggle/working/checkpoints")

    # ── Image Processing ───────────────────────────────────
    original_size: int = 256
    crop_size:     int = 180    # center crop post-rotation
    img_size:      int = 224    # final model input size
    pad_pixels:    int = 60     # reflection padding before rotation (60px for safe 45° margin)

    # ── Training ───────────────────────────────────────────
    backbone:     str = "convnext_tiny"
    pretrained:   bool  = True
    num_folds:    int   = 5
    epochs:       int   = 25
    batch_size:   int = 32
    lr:           float = 3e-4
    weight_decay: float = 1e-4
    warmup_epochs: int  = 2
    early_stop_patience: int = 7
    seed:         int   = 42

    # ── Class Imbalance ────────────────────────────────────
    # pos_weight DISABLED: it squashes predicted probabilities,
    # breaking the 0.5 validation cutoff and early stopping.
    # Imbalance is handled by OOF threshold sweep (Cell 7).
    use_pos_weight: bool  = False
    pos_weight:     float = 2854.0 / 5000.0  # ~0.571 (unused)

    # ── Azimuth MLP Head ───────────────────────────────────
    use_azimuth_mlp: bool = True
    azimuth_mlp_dim: int  = 32

    # ── Augmentation ───────────────────────────────────────
    scale_range:      Tuple[float, float] = (0.8, 1.0)  # proportion of image area (max must be ≤ 1.0)
    contrast_limit:   float = 0.15
    brightness_limit: float = 0.1
    blur_limit:       Tuple[int, int] = (3, 5)
    cutout_p:         float = 0.3
    clahe_p:          float = 0.3

    # ── Threshold Sweep ────────────────────────────────────
    thresh_lo:    float = 0.1
    thresh_hi:    float = 0.9
    thresh_steps: int   = 161

    # ── Device ─────────────────────────────────────────────
    @property
    def device(self):
        return "cuda" if torch.cuda.is_available() else "cpu"

    num_workers:  int = 0

cfg = CFG()

# Create output dirs
for d in [cfg.norm_train_dir, cfg.norm_eval_dir, cfg.ckpt_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Load & validate metadata
train_meta = pd.read_csv(cfg.input_dir / cfg.train_meta_path)
test_meta  = pd.read_csv(cfg.input_dir / cfg.test_meta_path)

print(f"Input directory: {cfg.input_dir}")
print(f"Train: {len(train_meta)} images | Test: {len(test_meta)} images")
print(f"Class 0 (Depth): {(train_meta['label']==0).sum()} ({(train_meta['label']==0).mean()*100:.1f}%)")
print(f"Class 1 (Rise):  {(train_meta['label']==1).sum()} ({(train_meta['label']==1).mean()*100:.1f}%)")
print(f"Azimuth range: [{train_meta['sun_azimuth_angle'].min():.1f}°, {train_meta['sun_azimuth_angle'].max():.1f}°]")
print(f"Device: {cfg.device}")

## Cell 2 — Exploratory Data Analysis

In [ ]:
# ============================================================
# CELL 2: Exploratory Data Analysis
# ============================================================
import matplotlib.pyplot as plt

# ── Sample images by class ────────────────────────────────
fig, axes = plt.subplots(2, 6, figsize=(22, 8))
fig.suptitle("Raw Samples — Before Azimuth Normalization", fontsize=14, y=1.02)

for cls_idx, cls_name in enumerate(["Class 0 — Depth (craters)", "Class 1 — Rise (mounds)"]):
    samples = train_meta[train_meta["label"] == cls_idx].sample(6, random_state=42)
    for i, (_, row) in enumerate(samples.iterrows()):
        img = cv2.imread(
            str(cfg.input_dir / cfg.train_img_folder / row["image_id"]),
            cv2.IMREAD_GRAYSCALE
        )
        ax = axes[cls_idx, i]
        ax.imshow(img, cmap="gray")
        ax.set_title(f"az={row['sun_azimuth_angle']:.0f}°", fontsize=9)
        ax.axis("off")
    axes[cls_idx, 0].set_ylabel(cls_name, fontsize=11, rotation=90, labelpad=50)

plt.tight_layout()
plt.savefig(cfg.work_dir / "eda_samples.png", dpi=100, bbox_inches="tight")
plt.show()

# ── Azimuth distribution per class ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for cls_idx, (ax, cls_name, color) in enumerate(
    zip(axes[:2], ["Depth (0)", "Rise (1)"], ["steelblue", "coral"])
):
    subset = train_meta[train_meta["label"] == cls_idx]
    ax.hist(subset["sun_azimuth_angle"], bins=36, alpha=0.7, color=color, edgecolor="white")
    ax.set_title(f"{cls_name} — n={len(subset)}")
    ax.set_xlabel("Sun Azimuth Angle (°)")
    ax.set_ylabel("Count")

# Test set azimuth
axes[2].hist(test_meta["sun_azimuth_angle"], bins=36, alpha=0.7, color="green", edgecolor="white")
axes[2].set_title(f"Test Set — n={len(test_meta)}")
axes[2].set_xlabel("Sun Azimuth Angle (°)")
plt.tight_layout()
plt.savefig(cfg.work_dir / "eda_azimuth.png", dpi=100)
plt.show()

## Cell 3 — Physics Normalization Engine (CPU Multiprocessing)> ⚡ **Run this WITHOUT GPU enabled** to preserve your 30-hour weekly quota.  > Takes ~5–10 min on 4 CPU cores for all 9,854 images.

In [ ]:
# ============================================================
# CELL 3: Azimuth Rotation — CPU Multiprocessing
# ============================================================
# ⚠️ RUN WITH GPU DISABLED to preserve quota
# ~5-10 min for 9,854 images on 4 CPU cores
# ============================================================
import cv2
import numpy as np
from multiprocessing import Pool
from functools import partial
from tqdm import tqdm

# CRITICAL: Disable OpenCV internal threading to prevent
# thread contention inside multiprocessing workers.
# Without this, OpenMP/pthreads inside cv2.warpAffine
# clash with Pool workers → CPU lockups and thrashing.
cv2.setNumThreads(0)

def normalize_single_image(args, src_dir, dst_dir, crop_size, img_size, pad_px):
    """
    Rotate one image by -azimuth with reflection padding,
    center-crop, and resize. This aligns all shadows to a
    canonical direction, eliminating topographic inversion.
    """
    image_id, azimuth = args
    dst_path = dst_dir / image_id
    
    # Skip if already processed (idempotent)
    if dst_path.exists():
        return image_id, True

    img = cv2.imread(str(src_dir / image_id), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return image_id, False

    # Step 1: Reflection padding → avoids black corners
    padded = cv2.copyMakeBorder(
        img, pad_px, pad_px, pad_px, pad_px, cv2.BORDER_REFLECT
    )
    ph, pw = padded.shape

    # Step 2: Rotate counter-clockwise by -azimuth
    center = (pw // 2, ph // 2)
    M = cv2.getRotationMatrix2D(center, -azimuth, 1.0)
    rotated = cv2.warpAffine(
        padded, M, (pw, ph),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT,
    )

    # Step 3: Center crop → removes any remaining edge artifacts
    cy, cx = ph // 2, pw // 2
    half = crop_size // 2
    cropped = rotated[cy - half : cy + half, cx - half : cx + half]

    # Step 4: Resize to model input dimensions
    resized = cv2.resize(
        cropped, (img_size, img_size), interpolation=cv2.INTER_LINEAR
    )

    cv2.imwrite(str(dst_path), resized)
    return image_id, True


def batch_normalize(meta_df, src_dir, dst_dir, cfg):
    """Process all images using multiprocessing pool."""
    dst_dir.mkdir(parents=True, exist_ok=True)

    already = set(p.name for p in dst_dir.glob("*.png"))
    todo = meta_df[~meta_df["image_id"].isin(already)]

    if len(todo) == 0:
        print(f"✓ All {len(meta_df)} images already normalized in {dst_dir}")
        return

    print(f"Processing {len(todo)} images ({len(already)} already done)")
    print(f"  Rotation: -azimuth | Pad: {cfg.pad_pixels}px reflect | "
          f"Crop: {cfg.crop_size}² | Resize: {cfg.img_size}²")

    worker = partial(
        normalize_single_image,
        src_dir=src_dir, dst_dir=dst_dir,
        crop_size=cfg.crop_size, img_size=cfg.img_size, pad_px=cfg.pad_pixels,
    )
    args_list = list(zip(todo["image_id"], todo["sun_azimuth_angle"]))

    with Pool(processes=cfg.num_workers) as pool:
        results = list(tqdm(
            pool.imap_unordered(worker, args_list),
            total=len(args_list), desc="Normalizing"
        ))

    ok = sum(1 for _, s in results if s)
    fail = sum(1 for _, s in results if not s)
    print(f"✓ Done: {ok} succeeded, {fail} failed")


# ── Process train + eval images ───────────────────────────
print("="*50)
print("TRAINING IMAGES")
print("="*50)
batch_normalize(
    train_meta,
    src_dir=cfg.input_dir / cfg.train_img_folder,
    dst_dir=cfg.norm_train_dir,
    cfg=cfg,
)

print("\n" + "="*50)
print("EVALUATION IMAGES")
print("="*50)
batch_normalize(
    test_meta,
    src_dir=cfg.input_dir / cfg.eval_img_folder,
    dst_dir=cfg.norm_eval_dir,
    cfg=cfg,
)

# ── Visual verification ───────────────────────────────────
print("\n── Rotation Verification ──")
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Before vs After Azimuth Normalization", fontsize=14)

samples = train_meta.sample(5, random_state=42)
for i, (_, row) in enumerate(samples.iterrows()):
    orig = cv2.imread(
        str(cfg.input_dir / cfg.train_img_folder / row["image_id"]),
        cv2.IMREAD_GRAYSCALE
    )
    norm = cv2.imread(
        str(cfg.norm_train_dir / row["image_id"]),
        cv2.IMREAD_GRAYSCALE
    )
    axes[0, i].imshow(orig, cmap="gray")
    axes[0, i].set_title(f"Raw (az={row['sun_azimuth_angle']:.0f}°)", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(norm, cmap="gray")
    axes[1, i].set_title(f"Normalized", fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Rotated+Cropped", fontsize=12)
plt.tight_layout()
plt.savefig(cfg.work_dir / "rotation_check.png", dpi=100)
plt.show()

print("\n🔄 NOW: Enable GPU in Kaggle settings → Accelerator → GPU P100/T4")
print("   The kernel will restart but /kaggle/working/ persists.")
print("   After restart: re-run Cell 0 and Cell 1, then continue to Cell 4.")

## Cell 4 — Dataset & Illumination-Safe Augmentations> After enabling GPU and restarting: **re-run Cells 0 + 1**, then continue here.**Excluded augmentations** (would corrupt labels):- ✗ `RandomRotate` — shifts normalized light vector- ✗ `VerticalFlip` — swaps depth↔rise (topographic inversion = the exact bug!)- ✗ `HorizontalFlip` — shifts lateral light direction- ✗ `ShiftScaleRotate` — rotation component is unsafe

In [ ]:
# ============================================================
# CELL 4: Dataset & Illumination-Safe Augmentations
# ============================================================
import albumentations as A
import torch
from torch.utils.data import Dataset

def get_train_transforms(cfg):
    return A.Compose([
        A.RandomResizedCrop(
            size=(cfg.img_size, cfg.img_size),
            scale=cfg.scale_range, ratio=(0.95, 1.05),
            interpolation=cv2.INTER_LINEAR, p=0.5,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=cfg.brightness_limit,
            contrast_limit=cfg.contrast_limit, p=0.5,
        ),
        A.GaussianBlur(blur_limit=cfg.blur_limit, p=0.2),
        A.GaussNoise(std_range=(0.01, 0.03), p=0.3),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=cfg.clahe_p),
        A.CoarseDropout(
            num_holes_range=(1, 3),
            hole_height_range=(16, 40), hole_width_range=(16, 40),
            fill=0, p=cfg.cutout_p,
        ),
    ])

def get_val_transforms(cfg):
    return A.Compose([A.Resize(cfg.img_size, cfg.img_size)])


class LunarDataset(Dataset):
    """Reads PRE-ROTATED images from /kaggle/working/norm_*."""

    MEAN = np.array([0.485, 0.456, 0.406])
    STD  = np.array([0.229, 0.224, 0.225])

    def __init__(self, df, img_dir, cfg, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.cfg = cfg
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(str(self.img_dir / row["image_id"]), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"{self.img_dir / row['image_id']}")

        if self.transform:
            img = self.transform(image=img)["image"]

        # Grayscale → 3-channel → normalize to ImageNet stats
        img = np.stack([img, img, img], axis=-1).astype(np.float32) / 255.0
        img = (img - self.MEAN) / self.STD
        img = torch.from_numpy(img.transpose(2, 0, 1)).float()

        # Azimuth → sin/cos encoding for MLP branch
        az = np.radians(row["sun_azimuth_angle"])
        az_feat = torch.tensor([np.sin(az), np.cos(az)], dtype=torch.float32)

        if self.is_test:
            return img, az_feat, row["image_id"]
        else:
            return img, az_feat, torch.tensor(row["label"], dtype=torch.float32)

print("✓ Dataset & transforms defined")

## Cell 5 — Hybrid CNN + Azimuth MLP Classifier```┌──────────────────────┐    ┌─────────────────┐│  EfficientNet-B0     │    │  Azimuth MLP     ││  (ImageNet pretrain) │    │  sin,cos → 32    ││  → 1280-d features   │    │                  │└──────────┬───────────┘    └────────┬────────┘           └──────── concat ────────┘                      │                Dropout(0.3)                Linear → 1                (BCEWithLogits)```

In [ ]:
# ============================================================
# CELL 5: Hybrid CNN + Azimuth MLP Classifier
# ============================================================
import torch.nn as nn
import timm

class LunarClassifier(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=cfg.pretrained, num_classes=0
        )
        feat_dim = self.backbone.num_features

        self.use_az = cfg.use_azimuth_mlp
        if self.use_az:
            self.az_mlp = nn.Sequential(
                nn.Linear(2, cfg.azimuth_mlp_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(0.1),
                nn.Linear(cfg.azimuth_mlp_dim, cfg.azimuth_mlp_dim),
                nn.ReLU(inplace=True),
            )
            head_in = feat_dim + cfg.azimuth_mlp_dim
        else:
            head_in = feat_dim

        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(head_in, 1),
        )

    def forward(self, images, az_feats):
        x = self.backbone(images)
        if self.use_az:
            x = torch.cat([x, self.az_mlp(az_feats)], dim=1)
        return self.head(x).squeeze(-1)

# Verify
_m = LunarClassifier(cfg).to(cfg.device)
_o = _m(torch.randn(2, 3, 224, 224).to(cfg.device), torch.randn(2, 2).to(cfg.device))
print(f"✓ Output shape: {_o.shape}")
print(f"  Total params:     {sum(p.numel() for p in _m.parameters()):,}")
print(f"  Trainable params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
del _m, _o; torch.cuda.empty_cache()

## Cell 6 — 5-Fold Stratified TrainingCore training with AMP (mixed precision), cosine annealing with delayed decay, and early stopping.  **Estimated time: ~15–25 min on P100/T4.**

In [ ]:
# ============================================================
# CELL 6: 5-Fold Stratified K-Fold Training
# ============================================================
import time, gc
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from tqdm import tqdm

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for images, az, labels in tqdm(loader, desc="  Train", leave=False):
        images = images.to(device, non_blocking=True)
        az     = az.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type="cuda"):
            logits = model(images, az)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(labels)
        all_preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        balanced_accuracy_score(all_labels, (np.array(all_preds) > 0.5).astype(int)),
    )


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    for images, az, labels in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(device, non_blocking=True)
        az     = az.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type="cuda"):
            logits = model(images, az)
            loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        all_preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    preds_arr = np.array(all_preds)
    labels_arr = np.array(all_labels)
    return (
        total_loss / len(loader.dataset),
        balanced_accuracy_score(labels_arr, (preds_arr > 0.5).astype(int)),
        preds_arr,
        labels_arr,
    )


def run_training(cfg):
    set_seed(cfg.seed)
    device = cfg.device
    train_meta = pd.read_csv(cfg.input_dir / cfg.train_meta_path)

    oof_preds  = np.zeros(len(train_meta))
    oof_labels = np.zeros(len(train_meta))
    fold_scores = []

    skf = StratifiedKFold(n_splits=cfg.num_folds, shuffle=True, random_state=cfg.seed)

    # Weighted BCE loss to handle class imbalance (Depth=36%, Rise=64%)
    if cfg.use_pos_weight:
        pw = torch.tensor([cfg.pos_weight], device=device)
        print(f"Using pos_weight={cfg.pos_weight:.3f} (down-weights majority Rise class)")
    else:
        pw = None

    for fold, (trn_idx, val_idx) in enumerate(skf.split(train_meta, train_meta["label"])):
        print(f"\n{'='*65}")
        print(f"  FOLD {fold+1}/{cfg.num_folds}  │  Train: {len(trn_idx)}  Val: {len(val_idx)}")
        print(f"{'='*65}")

        trn_ds = LunarDataset(
            train_meta.iloc[trn_idx], cfg.norm_train_dir, cfg,
            transform=get_train_transforms(cfg),
        )
        val_ds = LunarDataset(
            train_meta.iloc[val_idx], cfg.norm_train_dir, cfg,
            transform=get_val_transforms(cfg),
        )
        trn_loader = DataLoader(
            trn_ds, batch_size=cfg.batch_size, shuffle=True,
            num_workers=cfg.num_workers, pin_memory=True, drop_last=True,
        )
        val_loader = DataLoader(
            val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
            num_workers=cfg.num_workers, pin_memory=True,
        )

        model = LunarClassifier(cfg).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg.epochs - cfg.warmup_epochs, eta_min=1e-6
        )
        scaler = GradScaler("cuda")  # PyTorch 2.x requires explicit device

        best_acc = 0
        patience = 0
        ckpt = cfg.ckpt_dir / f"fold{fold}_best.pt"

        for ep in range(cfg.epochs):
            t0 = time.time()
            t_loss, t_acc = train_one_epoch(
                model, trn_loader, optimizer, criterion, scaler, device
            )
            v_loss, v_acc, v_preds, v_labels = validate(
                model, val_loader, criterion, device
            )
            # NOTE: This is "delayed decay" not a true warmup (LR stays
            # flat at 3e-4 for 2 epochs, then cosine decays). This is fine
            # for efficientnet_b0 on ~7k images — lets the new head settle.
            if ep >= cfg.warmup_epochs:
                scheduler.step()

            mark = ""
            if v_acc > best_acc:
                best_acc = v_acc
                patience = 0
                torch.save(model.state_dict(), ckpt)
                mark = " ✓ BEST"
            else:
                patience += 1

            print(
                f"  Ep {ep+1:02d}/{cfg.epochs} │ "
                f"T {t_loss:.4f}/{t_acc:.4f} │ "
                f"V {v_loss:.4f}/{v_acc:.4f} │ "
                f"LR {optimizer.param_groups[0]['lr']:.2e} │ "
                f"{time.time()-t0:.0f}s{mark}"
            )

            if patience >= cfg.early_stop_patience:
                print(f"  ⏹ Early stop (patience={cfg.early_stop_patience})")
                break

        # Best model → OOF predictions
        model.load_state_dict(torch.load(ckpt, weights_only=True))
        _, _, best_preds, best_labels = validate(model, val_loader, criterion, device)
        oof_preds[val_idx]  = best_preds
        oof_labels[val_idx] = best_labels
        fold_scores.append(best_acc)
        print(f"  Fold {fold+1} best: {best_acc:.4f}")

        del model, optimizer, scheduler, scaler
        gc.collect(); torch.cuda.empty_cache()

    # Save OOF
    oof_df = pd.DataFrame({
        "image_id": train_meta["image_id"],
        "label": oof_labels.astype(int),
        "pred_prob": oof_preds,
    })
    oof_df.to_csv(cfg.work_dir / "oof_predictions.csv", index=False)

    overall = balanced_accuracy_score(oof_labels, (oof_preds > 0.5).astype(int))
    print(f"\n{'='*65}")
    print(f"  OOF Balanced Accuracy: {overall:.4f}")
    print(f"  Per-fold: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"  Mean±Std: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"{'='*65}")
    return oof_preds, oof_labels


# ── RUN ────────────────────────────────────────────────────
oof_preds, oof_labels = run_training(cfg)

## Cell 7 — OOF Threshold SweepSweeps thresholds from 0.1 to 0.9 on out-of-fold predictions to find the cutoff that maximizes Balanced Accuracy.  This is how we handle the 64/36 class imbalance — cleaner than `pos_weight`.

In [ ]:
# ============================================================
# CELL 7: Threshold Sweep → Maximize Balanced Accuracy
# ============================================================
oof_df = pd.read_csv(cfg.work_dir / "oof_predictions.csv")
labels = oof_df["label"].values
probs  = oof_df["pred_prob"].values

thresholds = np.linspace(cfg.thresh_lo, cfg.thresh_hi, cfg.thresh_steps)
scores = [balanced_accuracy_score(labels, (probs >= t).astype(int)) for t in thresholds]

best_idx    = np.argmax(scores)
best_thresh = thresholds[best_idx]
best_score  = scores[best_idx]
default_score = balanced_accuracy_score(labels, (probs >= 0.5).astype(int))

print(f"Default (0.500):  Bal-Acc = {default_score:.4f}")
print(f"Optimal ({best_thresh:.3f}):  Bal-Acc = {best_score:.4f}")
print(f"Improvement:      +{(best_score - default_score)*100:.2f}%")

with open(cfg.work_dir / "optimal_threshold.txt", "w") as f:
    f.write(f"{best_thresh:.6f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, scores, lw=2, color="steelblue")
ax.axvline(best_thresh, color="red", ls="--", label=f"Optimal: {best_thresh:.3f}")
ax.axvline(0.5, color="gray", ls=":", alpha=0.6, label="Default: 0.500")
ax.set_xlabel("Threshold"); ax.set_ylabel("Balanced Accuracy")
ax.set_title(f"OOF Threshold Sweep — Best: {best_score:.4f} @ {best_thresh:.3f}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(cfg.work_dir / "threshold_sweep.png", dpi=100)
plt.show()

## Cell 8 — Inference & SubmissionRuns all 2,000 evaluation images through the 5-fold ensemble with soft voting and the optimized threshold.  Output: `/kaggle/working/submission.csv`

In [ ]:
# ============================================================
# CELL 8: Inference → submission.csv
# ============================================================
@torch.no_grad()
def predict_test(cfg):
    device = cfg.device
    test_meta = pd.read_csv(cfg.input_dir / cfg.test_meta_path)

    test_ds = LunarDataset(
        test_meta, cfg.norm_eval_dir, cfg,
        transform=get_val_transforms(cfg), is_test=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.batch_size * 2, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
    )

    all_fold_probs = []
    image_ids = None

    for fold in range(cfg.num_folds):
        ckpt = cfg.ckpt_dir / f"fold{fold}_best.pt"
        if not ckpt.exists():
            print(f"⚠️ Fold {fold} missing — skipping")
            continue

        model = LunarClassifier(cfg).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        model.eval()

        fold_probs, fold_ids = [], []
        for images, az, ids in tqdm(test_loader, desc=f"Fold {fold}"):
            images = images.to(device, non_blocking=True)
            az = az.to(device, non_blocking=True)
            with autocast(device_type="cuda" if device == "cuda" else "cpu"):
                logits = model(images, az)
            fold_probs.extend(torch.sigmoid(logits).cpu().numpy())
            fold_ids.extend(ids)

        all_fold_probs.append(np.array(fold_probs))
        if image_ids is None:
            image_ids = fold_ids

        del model; gc.collect(); torch.cuda.empty_cache()
        print(f"  Fold {fold}: {len(fold_probs)} predictions")

    # Soft voting across folds
    avg_probs = np.mean(all_fold_probs, axis=0)

    # Apply optimized threshold
    thresh_file = cfg.work_dir / "optimal_threshold.txt"
    threshold = float(thresh_file.read_text().strip()) if thresh_file.exists() else 0.5
    pred_labels = (avg_probs >= threshold).astype(int)

    # Build submission
    submission = pd.DataFrame({"image_id": image_ids, "label": pred_labels})
    sub_path = cfg.work_dir / f"submission_{cfg.backbone}.csv"
    submission.to_csv(sub_path, index=False)

    print(f"\n{'='*55}")
    print(f"  ✅ SUBMISSION: {sub_path}")
    print(f"  Rows:      {len(submission)}")
    print(f"  Threshold: {threshold:.3f}")
    print(f"  Class 0:   {(pred_labels==0).sum()} ({(pred_labels==0).mean()*100:.1f}%)")
    print(f"  Class 1:   {(pred_labels==1).sum()} ({(pred_labels==1).mean()*100:.1f}%)")
    print(f"  Folds:     {len(all_fold_probs)}")
    print(f"{'='*55}")

    # Format checks
    assert len(submission) == 2000, f"Expected 2000, got {len(submission)}"
    assert list(submission.columns) == ["image_id", "label"]
    assert set(submission["label"].unique()).issubset({0, 1})
    print("✓ Format verified: 2000 rows, [image_id, label], values ∈ {0, 1}")

    return submission

submission = predict_test(cfg)
submission.head(10)